<table style="width: 100%; border-collapse: collapse; border: none; background: #f8fafc; border-left: 6px solid #1e3a8a; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #0f172a; font-size: 2.1em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        Taller Hands-On: Cuando el Mundo Real Rompe los Supuestos 🚀
      </h1>
      <p style="margin: 6px 0 0 0; color: #1e3a8a; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Machine Learning
      </p>
      <p style="margin: 4px 0 0 0; color: #64748b; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #1e3a8a; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        Taller Práctico Evaluativo · Módulo 08
      </span><br>
      <span style="color: #64748b; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #2563eb; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Machine%20Learning/homeworks/08_Temas_Avanzados_Hands_On.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## Objetivos de Aprendizaje 🔎

Este taller consolida el **Módulo 08 (Temas avanzados de ML)** con **cuatro situaciones nuevas** que casi nunca aparecen en los datos de ejemplo: el mundo cambia, la clase importante es rara, faltan datos y el modelo es una caja negra. Todo se resuelve con NumPy, pandas y scikit-learn (sin `shap`, sin `imbalanced-learn`, sin descargas).

Al terminar serás capaz de:

1. **Detectar el *covariate shift*** con **validación adversaria** y corregirlo con **pesos por importancia** (cuaderno [00 · Capítulo 3 y conceptos avanzados](../08%20-%20Temas%20Avanzados%20de%20ML/00_Introduccion_Cap3_y_Conceptos_Avanzados.ipynb)).
2. Tratar un conjunto **muy desbalanceado** sin caer en la trampa de la fuga: pesos de clase, sobremuestreo **solo en entrenamiento** y **umbral** de decisión (cuaderno [03 · Datasets desbalanceados](../08%20-%20Temas%20Avanzados%20de%20ML/03_Datasets_Desbalanceados.ipynb)).
3. Manejar **datos faltantes** imputando **dentro del `Pipeline`** y comparando estrategias (cuaderno [04 · Datos faltantes](../08%20-%20Temas%20Avanzados%20de%20ML/04_Datos_Faltantes.ipynb)).
4. **Explicar** un modelo opaco con **importancia por permutación** y **dependencia parcial** programadas a mano (cuaderno [02 · Explicabilidad](../08%20-%20Temas%20Avanzados%20de%20ML/02_Explicabilidad_de_Modelos.ipynb)).

---
## Instrucciones

- Resuelve cada reto **de forma autónoma**: completa las celdas con `# TODO:` y ejecuta la celda de **autoverificación** que le sigue (los `assert` fallan con un mensaje claro hasta que el reto esté bien resuelto).
- No modifiques la celda de preparación ni las celdas de autoverificación.
- Antes de entregar usa *Restart & Run All*: el cuaderno debe correr de principio a fin (< 1 min) sin errores y sin ningún `# TODO` pendiente.
- Todo usa semillas fijas y datos sintéticos: no necesitas descargar nada.

---
### 📦 Preparación del entorno y de los datos

Ejecuta esta celda sin modificarla. Crea cuatro conjuntos **sintéticos**, uno por reto: `X_train`/`X_prod` (clientes de entrenamiento frente a los de producción), `X2`/`y2` (fraude, ≈ 5 % de casos), `X3`/`y3` (encuesta con valores faltantes) y `X4`/`y4` (fallas de máquinas).

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_predict, cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.metrics import (roc_auc_score, recall_score, f1_score, average_precision_score,
                             precision_recall_curve, accuracy_score)
import warnings; warnings.filterwarnings("ignore")

sig = lambda t: 1 / (1 + np.exp(-t))

# --- Reto 1: clientes de entrenamiento vs. producción (el ingreso de producción es MÁS ALTO) ---
rng = np.random.default_rng(21)
def _clientes(n, mu_ingreso):
    return pd.DataFrame({"edad": rng.normal(40, 10, n).clip(18, 75),
                         "ingreso": rng.normal(mu_ingreso, 1.0, n).clip(0.5),
                         "antiguedad": rng.gamma(4, 2, n)})
X_train, X_prod = _clientes(1500, 3.0), _clientes(800, 4.2)
y_train = (rng.random(1500) < sig(0.8 * (X_train.ingreso - 3) - 0.03 * (X_train.edad - 40) + 0.1 * (X_train.antiguedad - 8))).astype(int)   # 1 = paga a tiempo

# --- Reto 2: fraude muy desbalanceado ---
rng = np.random.default_rng(22)
c2 = ["monto", "hora", "distancia_ult", "n_tx_dia", "antig_cuenta", "pais_riesgo", "cambio_disp", "edad_cuenta"]
X2 = pd.DataFrame(rng.normal(size=(4000, 8)), columns=c2)
b2 = 1.3 * np.array([1.0, 0.7, 0.8, 0.5, -0.4, 0.6, 0.0, 0.0])
y2 = pd.Series((rng.random(4000) < sig(X2.to_numpy() @ b2 - 4.4)).astype(int), name="fraude")

# --- Reto 3: encuesta con faltantes (gasto y deuda faltan más cuando el ingreso es alto: MAR) ---
rng = np.random.default_rng(23)
n3 = 1200
c3 = ["ingreso", "gasto", "edad", "ahorro", "deuda", "antiguedad"]
X3 = pd.DataFrame(rng.normal(size=(n3, 6)), columns=c3)
y3 = pd.Series((rng.random(n3) < sig(1.0 * X3.ingreso + 0.9 * X3.gasto - 0.6 * X3.edad)).astype(int), name="compra")
X3.loc[rng.random(n3) < sig(2.0 * X3.ingreso - 1.0), "gasto"] = np.nan
X3.loc[rng.random(n3) < sig(1.5 * X3.ingreso - 1.8), "deuda"] = np.nan
X3.loc[rng.random(n3) < 0.12, "edad"] = np.nan
X3.loc[rng.random(n3) < 0.15, "ahorro"] = np.nan

# --- Reto 4: fallas de máquinas (temperatura manda, vibración aporta, el resto es ruido) ---
rng = np.random.default_rng(24)
c4 = ["temperatura", "vibracion", "presion", "horas_uso", "turno", "humedad"]
X4 = pd.DataFrame(rng.normal(size=(1500, 6)), columns=c4)
y4 = pd.Series((rng.random(1500) < sig(1.8 * X4.temperatura + 0.9 * X4.vibracion + 0.7 * X4.presion)).astype(int), name="falla")

print("Reto 1:", X_train.shape, X_prod.shape, "| Reto 2: fraude =", round(y2.mean(), 3), "| Reto 3: filas con algún faltante =",
      round(X3.isna().any(axis=1).mean(), 2), "| Reto 4:", X4.shape)

---
## Reto 1: ¿El Mundo de Producción se Parece al de Entrenamiento? 🌊 *(20 puntos)*

**Contexto.** Un banco entrenó su modelo con `X_train`, pero los clientes que llegan hoy (`X_prod`, **sin etiquetas** todavía) parecen distintos. Antes de confiar en las métricas hay que **demostrar el cambio** y, si existe, **corregirlo**.

**Enunciado.**
1. **Validación adversaria.** Apila `X_train` y `X_prod` en `X_adv` y crea `z` (0 = entrenamiento, 1 = producción). Con `cross_val_predict(...)`, `cv=5` y `method="predict_proba"`, estima `p_adv` (probabilidad de ser de producción) usando `make_pipeline(StandardScaler(), LogisticRegression())`. Calcula `auc_adv = roc_auc_score(z, p_adv)`.
2. Ajusta el mismo *pipeline* con **todos** los datos de `X_adv` y define `variable_sospechosa`: el nombre de la columna con mayor coeficiente **en valor absoluto**.
3. **Pesos por importancia.** Para las filas de entrenamiento (las primeras `len(X_train)` de `p_adv`) calcula $w_i = p_i/(1-p_i)$, recórtalos al rango $[0.05, 20]$ y normalízalos para que su media sea 1: `pesos`. Calcula el tamaño efectivo de muestra $n_{ef} = (\sum w_i)^2 / \sum w_i^2$ (`n_ef`) y la tasa de buenos pagadores `tasa_simple` (media de `y_train`) y `tasa_pond` (media ponderada con `pesos`).

**Criterios de aceptación.** `auc_adv > 0.75`; la variable sospechosa es la que realmente cambió; `pesos` positivos, media 1 y **mayores para los clientes de ingreso alto**; `n_ef < len(X_train)`; `tasa_pond > tasa_simple` (los clientes de producción, con más ingreso, pagan mejor).

In [ ]:
modelo_adv = make_pipeline(StandardScaler(), LogisticRegression())

# TODO 1: apila los datos, crea z y estima p_adv fuera de muestra (cross_val_predict, cv=5)
X_adv = None
z = None
p_adv = None
auc_adv = None

# TODO 2: ajusta modelo_adv con todo X_adv y busca la variable con mayor |coeficiente|
variable_sospechosa = None

# TODO 3: pesos por importancia solo para las filas de entrenamiento
n_tr = len(X_train)
pesos = None       # w = p/(1-p), recortado a [0.05, 20] y con media 1
n_ef = None
tasa_simple = None
tasa_pond = None

print("AUC adversaria:", auc_adv, "| variable sospechosa:", variable_sospechosa)
print("n efectivo:", n_ef, "de", n_tr, "| tasa simple:", tasa_simple, "| tasa ponderada:", tasa_pond)

In [ ]:
# ✅ Autoverificación del Reto 1 (no modifiques esta celda)
assert X_adv is not None and X_adv.shape == (2300, 3), "Reto 1: X_adv debe apilar X_train y X_prod (2300 filas, 3 columnas)."
assert z is not None and len(z) == 2300 and z.sum() == 800, "Reto 1: z vale 0 para las 1500 filas de entrenamiento y 1 para las 800 de producción."
assert p_adv is not None and p_adv.shape == (2300,) and 0 <= p_adv.min() and p_adv.max() <= 1, "Reto 1: p_adv son 2300 probabilidades (cross_val_predict con method='predict_proba', columna 1)."
assert auc_adv is not None and auc_adv > 0.75, f"Reto 1: el clasificador adversario debería distinguir bien (AUC > 0.75); obtuviste {auc_adv}."
assert variable_sospechosa == "ingreso", f"Reto 1: la variable que más delata el cambio es 'ingreso', no {variable_sospechosa!r}."
assert pesos is not None and pesos.shape == (1500,) and pesos.min() > 0, "Reto 1: pesos debe tener 1500 valores positivos (solo filas de entrenamiento)."
assert abs(pesos.mean() - 1) < 1e-6, "Reto 1: normaliza los pesos para que su media sea 1."
assert np.corrcoef(pesos, X_train["ingreso"])[0, 1] > 0.3, "Reto 1: los pesos deben crecer con el ingreso (producción tiene clientes de ingreso más alto)."
assert n_ef is not None and 100 < n_ef < 1500, f"Reto 1: el tamaño efectivo debe ser menor que 1500 pero razonable (obtuviste {n_ef}); revisa la fórmula (sum w)^2 / sum w^2."
assert tasa_simple is not None and tasa_pond is not None and tasa_pond > tasa_simple, "Reto 1: la tasa ponderada de buenos pagadores debe ser mayor que la simple."
print("✅ Reto 1 superado.")

---
## Reto 2: La Aguja en el Pajar (Fraude Raro) 🪡 *(20 puntos)*

**Contexto.** Solo ≈ 5 % de las transacciones de `X2` son fraude. Un modelo que dice «nunca hay fraude» acierta ≈ 95 % y no sirve para nada.

**Enunciado.**
1. Separa en `X2_tr`, `X2_te`, `y2_tr`, `y2_te` (30 % de prueba, `stratify=y2`, `random_state=0`).
2. `acc_dummy`: *accuracy* de prueba de `DummyClassifier(strategy="most_frequent")`. Entrena `LogisticRegression(max_iter=1000)` **plana** (con `StandardScaler` en un *pipeline*) y otra con `class_weight="balanced"`; guarda `recall_plano` y `recall_pesos` en prueba.
3. Implementa `sobremuestreo(X, y, rng)`: duplica **al azar con reemplazo** filas de la clase minoritaria hasta igualar a la mayoritaria y devuelve `X_b, y_b`. Aplícala **solo a los datos de entrenamiento** (`rng = np.random.default_rng(0)`), entrena el mismo *pipeline* y guarda `recall_over`.
4. **Umbral.** Con el modelo plano, obtén probabilidades de entrenamiento fuera de muestra (`cross_val_predict`, `cv=5`, `method="predict_proba"`), usa `precision_recall_curve` para escoger el umbral que **maximiza F1** (`umbral`) y calcula `f1_umbral` y `f1_05` (F1 en prueba con el umbral elegido y con 0.5). Guarda también `ap_test` (precisión promedio de prueba del modelo plano) y `prevalencia` (tasa de fraude de `y2_te`).

**Criterios de aceptación.** `acc_dummy > 0.9`; `recall_pesos > recall_plano` y `recall_over > recall_plano`; el sobremuestreo deja las clases iguales **y la prueba intacta**; `f1_umbral > f1_05`; `ap_test` al menos 3 veces la prevalencia.

In [ ]:
# TODO 1: partición estratificada 70 / 30
X2_tr = X2_te = y2_tr = y2_te = None

def pipe_lr(**kw):
    return make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, **kw))

# TODO 2: línea base y modelos plano / con pesos
acc_dummy = None
recall_plano = None
recall_pesos = None

def sobremuestreo(X, y, rng):
    """Duplica al azar filas de la clase minoritaria hasta igualar a la mayoritaria."""
    # TODO 3: devuelve X_b, y_b (arrays o DataFrame/Series) con las dos clases del mismo tamaño
    pass

recall_over = None   # entrenar SOLO con los datos de entrenamiento sobremuestreados

# TODO 4: umbral que maximiza F1 (con probabilidades fuera de muestra del entrenamiento)
umbral = None
f1_umbral = None
f1_05 = None
ap_test = None
prevalencia = None
print("Accuracy dummy:", acc_dummy, "| recall plano / pesos / over:", recall_plano, recall_pesos, recall_over)
print("Umbral:", umbral, "| F1 con umbral:", f1_umbral, "| F1 con 0.5:", f1_05, "| AP:", ap_test, "| prevalencia:", prevalencia)

In [ ]:
# ✅ Autoverificación del Reto 2 (no modifiques esta celda)
assert X2_tr is not None and X2_tr.shape == (2800, 8) and X2_te.shape == (1200, 8), "Reto 2: la partición debe ser 70 % / 30 % (2800 y 1200 filas)."
assert abs(y2_tr.mean() - y2_te.mean()) < 0.01, "Reto 2: la partición debe ser estratificada."
assert acc_dummy is not None and acc_dummy > 0.9, "Reto 2: la línea base 'most_frequent' debería tener accuracy > 0.9: ¡por eso accuracy engaña!"
assert recall_plano is not None and recall_pesos is not None and recall_pesos > recall_plano, "Reto 2: class_weight='balanced' debe aumentar el recall del fraude respecto del modelo plano."
_Xt, _yt = sobremuestreo(X2_tr.iloc[:500], y2_tr.iloc[:500], np.random.default_rng(1))
assert _Xt is not None and len(_Xt) == len(_yt), "Reto 2: sobremuestreo debe devolver X_b, y_b."
assert np.sum(np.asarray(_yt) == 1) == np.sum(np.asarray(_yt) == 0), "Reto 2: sobremuestreo debe dejar las dos clases con el mismo número de filas."
assert len(_yt) > 500, "Reto 2: debes duplicar filas de la clase minoritaria (el resultado es más grande que la entrada), no eliminar de la mayoritaria."
assert recall_over is not None and recall_over > recall_plano, "Reto 2: entrenar con datos sobremuestreados debe elevar el recall frente al modelo plano."
assert X2_te.shape == (1200, 8) and abs(y2_te.mean() - y2.mean()) < 0.01, "Reto 2: ¡la prueba no se toca! Sobremuestrea solo el entrenamiento."
assert umbral is not None and 0 < umbral < 0.5, f"Reto 2: con desbalance fuerte el umbral que maximiza F1 debe estar bajo 0.5 (obtuviste {umbral})."
assert f1_umbral is not None and f1_05 is not None and f1_umbral > f1_05, "Reto 2: el F1 de prueba con el umbral elegido debe superar al F1 con umbral 0.5."
assert ap_test is not None and prevalencia is not None and ap_test > 3 * prevalencia, "Reto 2: la precisión promedio debe ser al menos 3 veces la prevalencia."
print("✅ Reto 2 superado.")

---
## Reto 3: Encuesta con Huecos 🧩 *(20 puntos)*

**Contexto.** En `X3`, `gasto` y `deuda` faltan **más** cuando el ingreso es alto (mecanismo MAR) y `edad` y `ahorro` faltan al azar. Eliminar filas incompletas o imputar mal puede sesgar el modelo.

**Enunciado.**
1. `pct_faltante`: porcentaje de faltantes por columna (`Series`) y `frac_incompletas`: fracción de filas con **al menos un** faltante. Calcula `n_completas` = filas sin ningún faltante.
2. Construye **tres `Pipeline`** (el imputador **dentro** del *pipeline*, seguido de `StandardScaler` y `LogisticRegression(max_iter=1000)`), cada uno con una estrategia distinta: `"mediana"` (`SimpleImputer(strategy="median")`), `"indicador"` (`SimpleImputer(strategy="median", add_indicator=True)`) y `"knn"` (`KNNImputer(n_neighbors=5)`). Guárdalos en el diccionario `pipelines`.
3. Evalúa cada uno con `cross_val_score` sobre `X3`, `y3`, `scoring="roc_auc"` y `cv=StratifiedKFold(5, shuffle=True, random_state=0)`: guarda las medias en el diccionario `auc_por_estrategia` y el nombre de la mejor en `mejor`.

**Criterios de aceptación.** Más del 30 % de filas incompletas; tres *pipelines* cuyo **primer paso** es un imputador; el de indicador debe producir más columnas que las originales tras ajustarse; todas las AUC entre 0.7 y 1.

In [ ]:
# TODO 1: diagnóstico del faltante
pct_faltante = None      # % de faltantes por columna
frac_incompletas = None  # fracción de filas con al menos un faltante
n_completas = None

# TODO 2: tres pipelines (imputador DENTRO del pipeline)
pipelines = {
    "mediana": None,
    "indicador": None,
    "knn": None,
}

# TODO 3: AUC por validación cruzada y estrategia ganadora
cv3 = StratifiedKFold(5, shuffle=True, random_state=0)
auc_por_estrategia = {}
mejor = None
print(pct_faltante)
print("Filas incompletas:", frac_incompletas, "| completas:", n_completas)
print("AUC:", auc_por_estrategia, "| mejor:", mejor)

In [ ]:
# ✅ Autoverificación del Reto 3 (no modifiques esta celda)
assert pct_faltante is not None and list(pct_faltante.index) == list(X3.columns), "Reto 3: pct_faltante debe ser una Serie indexada por las columnas de X3."
assert pct_faltante["ingreso"] == 0 and pct_faltante["gasto"] > 20 and pct_faltante["ahorro"] > 8, "Reto 3: revisa el cálculo del porcentaje de faltantes (X3.isna().mean() * 100)."
assert frac_incompletas is not None and 0.3 < frac_incompletas < 0.9, f"Reto 3: la fracción de filas incompletas debe ser > 0.3 (obtuviste {frac_incompletas})."
assert n_completas == int(round((1 - frac_incompletas) * len(X3))), "Reto 3: n_completas debe ser consistente con frac_incompletas."
assert set(pipelines) == {"mediana", "indicador", "knn"} and all(isinstance(p, Pipeline) for p in pipelines.values()), "Reto 3: pipelines debe contener tres objetos Pipeline."
assert all(isinstance(p.steps[0][1], (SimpleImputer, KNNImputer)) for p in pipelines.values()), "Reto 3: el PRIMER paso de cada Pipeline debe ser el imputador (así se ajusta solo con el entrenamiento de cada pliegue)."
assert isinstance(pipelines["knn"].steps[0][1], KNNImputer) and pipelines["indicador"].steps[0][1].add_indicator, "Reto 3: 'knn' usa KNNImputer e 'indicador' usa SimpleImputer(add_indicator=True)."
_t = pipelines["indicador"].fit(X3, y3)
assert _t[:-1].transform(X3).shape[1] > X3.shape[1], "Reto 3: el imputador con indicador debe añadir columnas (una por cada variable con faltantes)."
assert set(auc_por_estrategia) == {"mediana", "indicador", "knn"}, "Reto 3: auc_por_estrategia debe tener las tres claves."
assert all(0.7 < v <= 1 for v in auc_por_estrategia.values()), "Reto 3: cada AUC debe estar entre 0.7 y 1; ¿usaste X3, y3 y scoring='roc_auc'?"
assert mejor == max(auc_por_estrategia, key=auc_por_estrategia.get), "Reto 3: mejor es la estrategia con mayor AUC medio."
print("✅ Reto 3 superado.")

---
## Reto 4: Abrir la Caja Negra 🔍 *(20 puntos)*

**Contexto.** Un bosque aleatorio predice fallas de máquinas. Mantenimiento quiere saber **qué variables importan** y **cómo** afectan el riesgo, pero sin instalar bibliotecas nuevas: lo programarás tú.

**Enunciado.** Se entrena `bosque` (ya ajustado en la celda de abajo con `X4_tr`, `y4_tr`).
1. Implementa `importancia_permutacion(modelo, X, y, n_rep, rng)`: para cada columna, baraja sus valores `n_rep` veces (`rng.permutation`), mide cuánto **cae el AUC** respecto del AUC original y devuelve una `Series` (índice = columnas) con la **caída media**. Calcula `imp = importancia_permutacion(bosque, X4_te, y4_te, 5, np.random.default_rng(0))`.
2. Implementa `dependencia_parcial(modelo, X, columna, rejilla)`: para cada valor de la rejilla, fija esa columna en ese valor **en todas las filas** y devuelve la probabilidad media predicha de falla (un arreglo). Calcula `pdp_temp` para `temperatura` con `rejilla = np.linspace(-2, 2, 9)`.
3. Escribe en `variable_top` el nombre de la variable más importante y en `variables_ruido` la lista de variables cuya importancia es **menor que 0.01**.

**Criterios de aceptación.** `imp` tiene una entrada por columna; `temperatura` encabeza el ranking; `horas_uso`, `turno` y `humedad` están entre las de ruido; `pdp_temp` es creciente (correlación > 0.9 con la rejilla) y está en $[0, 1]$.

In [ ]:
X4_tr, X4_te, y4_tr, y4_te = train_test_split(X4, y4, test_size=0.3, random_state=0)
bosque = RandomForestClassifier(n_estimators=60, min_samples_leaf=5, random_state=0, n_jobs=1).fit(X4_tr, y4_tr)

def importancia_permutacion(modelo, X, y, n_rep, rng):
    """Caída media del AUC al barajar cada columna. Devuelve una Series indexada por columna."""
    # TODO 1: AUC base y, por cada columna, n_rep barajados (trabaja sobre una copia de X)
    pass

def dependencia_parcial(modelo, X, columna, rejilla):
    """Probabilidad media predicha de falla cuando 'columna' se fija en cada valor de la rejilla."""
    # TODO 2: para cada valor, copia X, asigna la columna y promedia predict_proba[:, 1]
    pass

imp = None   # TODO: importancia_permutacion(bosque, X4_te, y4_te, 5, np.random.default_rng(0))
rejilla = np.linspace(-2, 2, 9)
pdp_temp = None   # TODO: dependencia_parcial(bosque, X4_te, "temperatura", rejilla)

# TODO 3: variable más importante y variables con importancia < 0.01
variable_top = None
variables_ruido = []
print(imp)
print("PDP:", pdp_temp, "| top:", variable_top, "| ruido:", variables_ruido)

In [ ]:
# ✅ Autoverificación del Reto 4 (no modifiques esta celda)
assert imp is not None and isinstance(imp, pd.Series) and list(imp.index) == list(X4.columns), "Reto 4: imp debe ser una Series con una entrada por cada columna de X4."
assert imp.idxmax() == "temperatura", f"Reto 4: la variable más importante debe ser 'temperatura' (obtuviste {imp.idxmax()!r}); revisa que midas la CAÍDA del AUC al barajar."
assert imp["temperatura"] > 0.1, "Reto 4: la caída del AUC por barajar 'temperatura' debería ser grande (> 0.1)."
assert imp[["horas_uso", "turno", "humedad"]].abs().max() < imp["temperatura"] / 3, "Reto 4: las variables de ruido deben tener una caída mucho menor que 'temperatura'."
_x = X4_te.copy(); _x_ok = importancia_permutacion(bosque, _x, y4_te, 2, np.random.default_rng(1))
assert _x.equals(X4_te), "Reto 4: no modifiques X en el sitio; trabaja sobre una copia al barajar."
assert pdp_temp is not None and len(pdp_temp) == len(rejilla), "Reto 4: pdp_temp debe tener un valor por punto de la rejilla."
assert 0 <= pdp_temp.min() and pdp_temp.max() <= 1, "Reto 4: pdp_temp son probabilidades medias (entre 0 y 1)."
assert np.corrcoef(rejilla, pdp_temp)[0, 1] > 0.9, "Reto 4: a mayor temperatura, mayor probabilidad media de falla: pdp_temp debe ser creciente."
assert variable_top == "temperatura", "Reto 4: variable_top debe ser 'temperatura'."
assert set(variables_ruido) == {"horas_uso", "turno", "humedad"}, f"Reto 4: variables_ruido debe ser exactamente horas_uso, turno y humedad (importancia < 0.01); obtuviste {sorted(variables_ruido)}."
print("✅ Reto 4 superado.")

---
## Reto 5: Conclusiones 🧾 *(20 puntos)*

Escribe **5 a 8 líneas** interpretando tus resultados, con tus números:

- ¿Qué AUC obtuvo el clasificador adversario y qué variable delató el cambio? ¿Qué implica para confiar en las métricas del entrenamiento y qué costo tiene reponderar (tamaño efectivo)? (Reto 1)
- ¿Por qué la *accuracy* de la línea base no sirve con fraude raro y cuál de los remedios (pesos, sobremuestreo, umbral) te pareció mejor? ¿Por qué el sobremuestreo debe hacerse solo en entrenamiento? (Reto 2)
- ¿Qué estrategia de imputación ganó y qué ventaja tiene imputar dentro del `Pipeline` frente a imputar todo el conjunto antes de validar? (Reto 3)
- ¿Qué te dicen la importancia por permutación y la dependencia parcial, y qué **no** te dicen (causalidad, variables correlacionadas)? (Reto 4)

---
#### ✍️ Tus conclusiones

_Escribe aquí tus 5 a 8 líneas. Reemplaza este texto por tu análisis._

---
### 🧠 Rúbrica de evaluación

| Criterio | Puntos |
|---|---|
| Reto 1: validación adversaria, variable sospechosa y pesos por importancia | 20 |
| Reto 2: línea base, pesos, sobremuestreo sin fuga y umbral por F1 | 20 |
| Reto 3: diagnóstico de faltantes e imputación dentro del `Pipeline` | 20 |
| Reto 4: importancia por permutación y dependencia parcial a mano | 20 |
| Reto 5: conclusiones numéricas, con límites de interpretación | 20 |
| **Total** | **100** |

### ✅ Checklist de entrega

- [ ] No queda ningún `# TODO` ni variable en `None`.
- [ ] Las cuatro celdas de autoverificación imprimen «✅ Reto N superado».
- [ ] *Restart & Run All* corre de principio a fin sin errores.
- [ ] La celda de conclusiones está escrita con tus propias palabras y con números.
- [ ] No modificaste la celda de preparación ni las autoverificaciones.

---
<div align="center">
  <p style="font-size: 0.9em; color: #64748b;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Asignatura: Machine Learning</i>
  </p>
</div>